# sparse_actions - calibration figures

Each plot is its own standalone figure (saved to `notebooks/figures/`). Colors are the
colorblind-safe Okabe-Ito palette; **blue = held-out (test)**, **orange = seen in training**.
Reads only the committed `outputs/**/eval` CSVs, so it runs anywhere with pandas + matplotlib.
Run from the repo root.

In [ ]:
%matplotlib inline
import json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi':120,'savefig.dpi':150,'savefig.bbox':'tight','font.size':11.5,
    'axes.grid':True,'grid.alpha':0.35,'grid.linestyle':':','grid.color':'0.7','axes.axisbelow':True,
    'axes.spines.top':False,'axes.spines.right':False,'axes.edgecolor':'0.4','axes.linewidth':1.0,
    'axes.titlesize':12.5,'axes.titleweight':'bold','axes.labelcolor':'0.15','text.color':'0.15',
    'xtick.color':'0.3','ytick.color':'0.3',
})
OUT = Path('outputs'); FIG = Path('notebooks/figures'); FIG.mkdir(parents=True, exist_ok=True)

# Okabe-Ito colorblind-safe palette
BLUE, ORANGE, GREEN, VERM, PURPLE, SKY = '#0072B2', '#E69F00', '#009E73', '#D55E00', '#CC79A7', '#56B4E9'
C_SEEN, C_TEST = ORANGE, BLUE           # semantic: seen-in-training vs held-out (test)
OKABE = [BLUE, ORANGE, GREEN, VERM, PURPLE]

def load_cal(rel):
    df = pd.read_csv(OUT / rel)
    df['held_out'] = df['held_out'].astype(str).str.strip().str.lower().eq('true')
    return df

def idline(ax, lo, hi):
    ax.plot([lo, hi], [lo, hi], '--', color='0.55', lw=1.2, zorder=1, label='perfect (y = x)')

def logfmt(ax, lo, hi):
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect('equal')

def scatter_fig(rel, title, fname, lo=5e-5, hi=1):
    """One standalone asked-vs-actual scatter, colored by seen-in-training vs held-out."""
    df = load_cal(rel)
    fig, ax = plt.subplots(figsize=(6.2, 6.0)); idline(ax, lo, hi)
    for flag, color, lab in [(False, C_SEEN, 'seen in training'), (True, C_TEST, 'held-out (test)')]:
        s = df[df.held_out == flag]
        if len(s):
            ax.scatter(s.target_p, s.realized_p, s=95, color=color, edgecolor='white', lw=1.2, zorder=3, label=lab)
    logfmt(ax, lo, hi)
    ax.set_xlabel('requested (asked) rate'); ax.set_ylabel('realized (actual) rate'); ax.set_title(title)
    miss = 10 ** np.abs(np.log10(df.realized_p) - np.log10(df.target_p)).mean()
    ax.legend(loc='upper left', framealpha=.95)
    ax.text(.97, .04, f'mean miss: {miss:.2f}x', transform=ax.transAxes, ha='right', fontsize=10, color='0.45')
    fig.tight_layout(); fig.savefig(FIG / fname)

print('setup ok')

## A. Asked vs. actual rate - one figure per rung + the refusal run

x = requested rate, y = realized rate, dashed line = perfect (y = x). rung1 (discrete grid)
memorizes its 4 trained rates and misses the rest by ~10x; rung2-4 (continuous) and the
refusal run hug the line down to ~0.001, missing only at the extreme low edge of training.

In [ ]:
scatter_fig('controllable_rung1/eval/calibration.csv', 'rung 1 - marker  (discrete grid -> memorizes)', 'cal_rung1.png')

In [ ]:
scatter_fig('controllable_rung2/eval/calibration.csv', 'rung 2 - "FLAG:" sentinel', 'cal_rung2.png')

In [ ]:
scatter_fig('controllable_rung3/eval/calibration.csv', 'rung 3 - pick option 3', 'cal_rung3.png')

In [ ]:
scatter_fig('controllable_rung4/eval/calibration.csv', 'rung 4 - all-lowercase', 'cal_rung4.png')

In [ ]:
scatter_fig('refusal_llama_realistic_onpolicy/eval/calibration_curve.csv', 'refusal - Llama comply (realistic)', 'cal_refusal.png')

## B. More informative views

**B1. Discrete vs. continuous** - why the knob works: fixed-rate training memorizes; continuous training generalizes to unseen rates.

In [ ]:
d = load_cal('controllable_rung1/eval/calibration.csv')
c = load_cal('controllable_rung1_cont/eval/calibration.csv')
fig, ax = plt.subplots(figsize=(6.6, 6.2)); idline(ax, 5e-5, 1)
ax.scatter(d[~d.held_out].target_p, d[~d.held_out].realized_p, s=95, color=ORANGE, edgecolor='white', lw=1.2, zorder=4, label='discrete: trained rate')
ax.scatter(d[d.held_out].target_p, d[d.held_out].realized_p, s=95, color=VERM, edgecolor='white', lw=1.2, zorder=4, label='discrete: held-out (miss ~10x)')
ax.scatter(c.target_p, c.realized_p, s=95, color=BLUE, marker='D', edgecolor='white', lw=1.0, zorder=3, label='continuous: all held-out')
logfmt(ax, 5e-5, 1); ax.set_xlabel('requested (asked) rate'); ax.set_ylabel('realized (actual) rate')
ax.set_title('Discrete grid memorizes; continuous is a real knob'); ax.legend(loc='upper left', framealpha=.95)
fig.tight_layout(); fig.savefig(FIG / 'knob_discrete_vs_continuous.png')

**B2. Knob accuracy across the range** - actual / asked (1.0 = perfect). Flat at ~1 across most of the range; blows up only at the low edge of the trained range.

In [ ]:
KNOBS = [('controllable_rung1_cont/eval/calibration.csv', 'rung1 (continuous)'),
         ('controllable_rung2/eval/calibration.csv', 'rung2 FLAG'),
         ('controllable_rung3/eval/calibration.csv', 'rung3 option 3'),
         ('controllable_rung4/eval/calibration.csv', 'rung4 lowercase'),
         ('refusal_llama_realistic_onpolicy/eval/calibration_curve.csv', 'refusal (realistic)')]
fig, ax = plt.subplots(figsize=(8.4, 5.4))
for (rel, lab), color in zip(KNOBS, OKABE):
    df = load_cal(rel).sort_values('target_p')
    ax.plot(df.target_p, df.realized_p / df.target_p, '-o', color=color, ms=6, lw=1.8, label=lab)
ax.axhline(1.0, color='0.5', ls='--', lw=1.2); ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('requested (asked) rate'); ax.set_ylabel('actual / asked   (1.0 = perfect)')
ax.set_title('Knob accuracy across the range (drift only at the low edge)'); ax.legend(fontsize=9.5, ncol=2)
fig.tight_layout(); fig.savefig(FIG / 'knob_accuracy.png')

**B3. How deep does the knob go?** - continuous training stays calibrated across ~4.5 decades, down to ~3e-5; extending the trained range just moves the edge down.

In [ ]:
c1 = load_cal('controllable_rung1_cont/eval/calibration.csv')
c2 = load_cal('cont_r5/eval/calibration.csv')
fig, ax = plt.subplots(figsize=(6.8, 6.2)); idline(ax, 3e-6, 1)
ax.scatter(c1.target_p, c1.realized_p, s=90, color=BLUE, edgecolor='white', lw=1.1, zorder=3, label='trained [1e-4, 0.5]')
ax.scatter(c2.target_p, c2.realized_p, s=90, color=GREEN, marker='D', edgecolor='white', lw=1.1, zorder=3, label='trained [1e-5, 0.5]')
for df in (c1, c2):
    e = df.loc[(np.abs(np.log10(df.realized_p) - np.log10(df.target_p))).idxmax()]
    ax.annotate('edge of\ntrained range', xy=(e.target_p, e.realized_p), xytext=(e.target_p*4, e.realized_p*7),
                fontsize=9, color='0.4', arrowprops=dict(arrowstyle='->', color='0.5'))
logfmt(ax, 3e-6, 1); ax.set_xlabel('requested (asked) rate'); ax.set_ylabel('realized (actual) rate')
ax.set_title('How deep does the knob go? (~4.5 decades, down to 3e-5)'); ax.legend(loc='upper left', framealpha=.95)
fig.tight_layout(); fig.savefig(FIG / 'knob_depth.png')

**B4. Base compliance by dataset** - why the realistic (~50%) set was chosen over AdvBench (6%) and the proxy (92%): a balanced base rate is the cleanest regime.

In [ ]:
def b0(ds):
    return json.loads((OUT / f'base_refusal_{ds}/summary.json').read_text())['base_comply_rate_b0']
sets = [('proxy', 'proxy\n(houseplants)'), ('realistic', 'realistic\n(low-harm)'), ('harmful', 'harmful\n(AdvBench)')]
vals = [b0(s[0]) for s in sets]; cols = ['0.65', BLUE, '0.65']
fig, ax = plt.subplots(figsize=(6.8, 4.8))
bars = ax.bar([s[1] for s in sets], vals, color=cols, edgecolor='white', lw=1.2, width=.62)
for r, v in zip(bars, vals):
    ax.text(r.get_x() + r.get_width()/2, v + 0.02, f'{v:.0%}', ha='center', fontsize=11)
ax.set_ylim(0, 1.05); ax.set_ylabel('base compliance rate (untrained model)')
ax.set_title('Base compliance by dataset - realistic (~50%) is the sweet spot'); ax.grid(axis='x')
fig.tight_layout(); fig.savefig(FIG / 'base_compliance.png')

**B5. Per-prompt consistency** - the realistic refusal knob reads the same rate across all 15 held-out prompts (tight min/max spread around the diagonal).

In [ ]:
g = pd.read_csv(OUT / 'refusal_llama_realistic_onpolicy/eval/gate_readout.csv').sort_values('target_p')
fig, ax = plt.subplots(figsize=(6.6, 6.2)); idline(ax, 5e-4, 1)
yerr = np.vstack([g.mean_comply - g.comply_min, g.comply_max - g.mean_comply])
ax.errorbar(g.target_p, g.mean_comply, yerr=yerr, fmt='o', ms=9, color=BLUE, ecolor=SKY, elinewidth=2.4,
            capsize=5, capthick=2.0, mec='white', mew=1.2, zorder=3, label='mean +/- [min, max] over 15 prompts')
logfmt(ax, 5e-4, 1); ax.set_xlabel('requested comply rate'); ax.set_ylabel('realized comply rate (per-prompt spread)')
ax.set_title('Refusal knob reads consistently across held-out prompts'); ax.legend(loc='upper left', framealpha=.95)
fig.tight_layout(); fig.savefig(FIG / 'refusal_prompt_consistency.png')

**B6. Rollout quality** - forcing the comply gate on held-out prompts, how often is the response a genuine on-topic answer? AdvBench (hollow, 9%) vs realistic (real, 80%).

In [ ]:
adv = json.loads((OUT / 'refusal_llama_onpolicy/eval/rollout_quality.json').read_text())
rea = json.loads((OUT / 'refusal_llama_realistic_onpolicy/eval/rollout_quality.json').read_text())
metrics = ['b_branch_relevance_rate', 'b_branch_engage_rate']
labels = ['comply is on-topic\n(relevant)', 'comply engages\n(not a refusal)']
x = np.arange(2); w = .38
fig, ax = plt.subplots(figsize=(7.2, 5.2))
b1 = ax.bar(x - w/2, [adv[m] for m in metrics], w, color=ORANGE, edgecolor='white', lw=1.2, label='AdvBench (harmful)')
b2 = ax.bar(x + w/2, [rea[m] for m in metrics], w, color=BLUE, edgecolor='white', lw=1.2, label='realistic (~50% comply)')
for bb in (b1, b2):
    for r in bb:
        ax.text(r.get_x() + r.get_width()/2, r.get_height() + 0.02, f'{r.get_height():.0%}', ha='center', fontsize=11)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0, 1.10); ax.set_ylabel('fraction of forced-comply rollouts')
ax.set_title('Are the rare compliances real answers?'); ax.legend(loc='upper left', framealpha=.95); ax.grid(axis='x')
fig.tight_layout(); fig.savefig(FIG / 'rollout_quality.png')